# Lab 5 - Time Series Analysis: Air Passengers

This notebook analyzes the AirPassengersDates.csv dataset from ./datasets/, covering date/time manipulation, visualization, aggregation, outlier detection, resampling, lag analysis, and autocorrelation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf

# Load data
df = pd.read_csv("./datasets/AirPassengersDates.csv")
print(df.head())
print(df.info())

## Section 1 – Date/Time Manipulation

Convert 'Date' column to datetime and extract Month, Day, and Day Name.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Day_Name'] = df['Date'].dt.day_name()
print(df.head())
print(df.info())

## Section 2 – Basic Time Series Visualization

Line chart of passengers over time.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['#Passengers'])
plt.title("Air Passengers Over Time")
plt.xlabel("Date")
plt.ylabel("Number of Passengers")
plt.grid(True)
plt.show()

## Section 3 – Aggregation and Bar Plot

Group by month and plot total passengers per month using seaborn.

In [ ]:
monthly = df.groupby('Month')['#Passengers'].sum().reset_index()
plt.figure(figsize=(10, 5))
sns.barplot(data=monthly, x='Month', y='#Passengers')
plt.title("Total Passengers per Month")
plt.show()

## Section 4 – Mean and Standard Deviation

Compute mean and std, visualize with horizontal reference lines.

In [ ]:
mean_pass = df['#Passengers'].mean()
std_pass = df['#Passengers'].std()
print(f"Mean: {mean_pass:.2f}, Std: {std_pass:.2f}")

plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['#Passengers'], label='Passengers')
plt.axhline(mean_pass, color='red', linestyle='--', label=f'Mean: {mean_pass:.2f}')
plt.axhline(mean_pass + std_pass, color='green', linestyle='--', label=f'Mean + Std: {mean_pass + std_pass:.2f}')
plt.axhline(mean_pass - std_pass, color='green', linestyle='--', label=f'Mean - Std: {mean_pass - std_pass:.2f}')
plt.title("Passengers with Mean and Standard Deviation")
plt.xlabel("Date")
plt.ylabel("#Passengers")
plt.legend()
plt.grid(True)
plt.show()

## Section 5 – Outlier Detection

Compute Z-scores to identify outliers.

In [ ]:
df['Z_Score'] = (df['#Passengers'] - mean_pass) / std_pass
df['Absolute_Z_Score'] = np.abs(df['Z_Score'])
print(df.nlargest(10, 'Absolute_Z_Score')[['Date', '#Passengers', 'Z_Score', 'Absolute_Z_Score']])

## Section 6 – Outlier Visualization

Highlight outliers (Absolute Z-Score > 2) on the time series plot.

In [ ]:
outliers = df[df['Absolute_Z_Score'] > 2]

plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['#Passengers'], label='All Passengers')
plt.scatter(outliers['Date'], outliers['#Passengers'], color='red', s=50, label='Outliers')
plt.title("Air Passengers with Outliers")
plt.xlabel("Date")
plt.ylabel("#Passengers")
plt.legend()
plt.grid(True)
plt.show()

## Section 7 – Resampling

Set Date as index and demonstrate upsampling (daily) and downsampling (yearly).

In [ ]:
df.set_index('Date', inplace=True)

# Upsampling to daily
daily = df.resample('D').asfreq().interpolate(method='linear')
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['#Passengers'], alpha=0.7, label='Original')
plt.plot(daily.index, daily['#Passengers'], linestyle='--', label='Upsampled Daily')
plt.title("Upsampling to Daily Frequency")
plt.legend()
plt.grid(True)
plt.show()

# Downsampling to yearly
yearly = df['#Passengers'].resample('Y').mean()
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['#Passengers'], alpha=0.5, label='Original')
plt.plot(yearly.index, yearly.values, marker='o', label='Yearly Average')
plt.title("Downsampling to Yearly Frequency")
plt.legend()
plt.grid(True)
plt.show()

## Section 8 – Lag Analysis (shift)

Create lagged features using shift() and tshift().

In [ ]:
df['#Passengers_Shift'] = df['#Passengers'].shift(periods=1)
df['#Passengers_tShift'] = df['#Passengers'].tshift(periods=1)
print(df.head())

plt.figure(figsize=(12, 6))
plt.plot(df.index, df['#Passengers'], label='#Passengers')
plt.plot(df.index, df['#Passengers_Shift'], label='#Passengers_Shift')
plt.plot(df.index, df['#Passengers_tShift'], label='#Passengers_tShift')
plt.title("Shift vs tShift")
plt.legend()
plt.grid(True)
plt.show()

## Section 9 – Autocorrelation

Plot the Autocorrelation Function (ACF) to analyze temporal dependencies.

In [ ]:
plt.figure(figsize=(10, 5))
plot_acf(df['#Passengers'].dropna(), lags=30)
plt.title("Autocorrelation Function (ACF)")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.show()